# Paper Tables + Figures Pack (Native Notebook)

Build final thesis tables and figures from frozen RL outputs and advisor experiment outputs. No shell commands are used.

## 1) Setup

In [1]:
from pathlib import Path
from datetime import datetime
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'coup').exists():
    fallback = Path('/Users/crishuynh/Documents/SoftwareProject/coup_detection')
    if (fallback / 'coup').exists():
        ROOT = fallback
os.chdir(ROOT)
print('Repository root:', ROOT)


Repository root: /Users/crishuynh/Documents/SoftwareProject/coup_detection


## 2) Configure source paths (edit if needed)

In [2]:
ADVISOR_SOURCE = ROOT / 'data' / 'research' / 'decision'
RL_SOURCE = ROOT / 'data' / 'research' / 'frozen_runs' / '20260409_135754' / 'dqn_rl_fit'
PAPER_DIR = ROOT / 'data' / 'research' / 'paper_artifacts' / datetime.now().strftime('%Y%m%d_%H%M%S')
PAPER_DIR.mkdir(parents=True, exist_ok=True)
print('Advisor source:', ADVISOR_SOURCE)
print('RL source:', RL_SOURCE)
print('Paper output:', PAPER_DIR)


Advisor source: /Users/crishuynh/Documents/SoftwareProject/coup_detection/data/research/decision
RL source: /Users/crishuynh/Documents/SoftwareProject/coup_detection/data/research/frozen_runs/20260409_135754/dqn_rl_fit
Paper output: /Users/crishuynh/Documents/SoftwareProject/coup_detection/data/research/paper_artifacts/20260412_173937


## 3) Validate required files

In [3]:
required = {
    'advisor_multiseed_thesis': ADVISOR_SOURCE / 'advisor_multiseed_thesis_table.csv',
    'advisor_bot_mix': ADVISOR_SOURCE / 'advisor_bot_mix_comparison.csv',
    'advisor_weighted_rank': ADVISOR_SOURCE / 'advisor_condition_weighted_ranking.csv',
    'rl_summary': RL_SOURCE / 'summary.json',
    'rl_history': RL_SOURCE / 'train_episode_history.csv',
    'rl_action_counts': RL_SOURCE / 'train_action_counts.json',
    'rl_state_summary': RL_SOURCE / 'train_state_summary.json',
    'rl_trace': RL_SOURCE / 'dqn_policy_rollout_trace.csv',
}

# Backfill weighted ranking if the source run did not export it.
if not required['advisor_weighted_rank'].exists() and required['advisor_bot_mix'].exists():
    botmix_df = pd.read_csv(required['advisor_bot_mix'])
    if {'condition', 'bot_mix', 'outcome_accuracy_mean', 'challenge_f1_mean'}.issubset(botmix_df.columns):
        rank_df = botmix_df.copy()
        rank_df['weighted_score'] = 0.6 * rank_df['outcome_accuracy_mean'] + 0.4 * rank_df['challenge_f1_mean']
        rank_df = rank_df.sort_values('weighted_score', ascending=False)
        rank_df.to_csv(required['advisor_weighted_rank'], index=False)
        print('Backfilled advisor weighted ranking at', required['advisor_weighted_rank'])

missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')
print('All required files found.')


All required files found.


## 4) Load core data

In [4]:
advisor_thesis_df = pd.read_csv(required['advisor_multiseed_thesis'])
advisor_botmix_df = pd.read_csv(required['advisor_bot_mix'])
advisor_rank_df = pd.read_csv(required['advisor_weighted_rank'])
rl_history_df = pd.read_csv(required['rl_history'])
rl_trace_df = pd.read_csv(required['rl_trace'])
rl_summary = json.loads(required['rl_summary'].read_text())
rl_action_counts = json.loads(required['rl_action_counts'].read_text())
rl_state_summary = json.loads(required['rl_state_summary'].read_text())
print('Loaded advisor thesis rows:', len(advisor_thesis_df))
print('Loaded botmix rows:', len(advisor_botmix_df))
print('Loaded rl history rows:', len(rl_history_df))
print('Loaded rl trace rows:', len(rl_trace_df))


Loaded advisor thesis rows: 5
Loaded botmix rows: 2
Loaded rl history rows: 100
Loaded rl trace rows: 1462


## 5) Build final thesis tables

In [5]:
table_advisor_main = advisor_thesis_df.copy()
table_advisor_main.to_csv(PAPER_DIR / 'table_advisor_main.csv', index=False)

table_botmix = advisor_botmix_df.copy()
table_botmix.to_csv(PAPER_DIR / 'table_botmix_comparison.csv', index=False)

table_weighted = advisor_rank_df.copy()
table_weighted.to_csv(PAPER_DIR / 'table_weighted_ranking.csv', index=False)

bench = rl_summary.get("benchmark", {})
policy_rows = []
for name in ['dqn_policy', 'random_policy', 'behavior_cloning']:
    if name in bench:
        policy_rows.append({
            'policy': name,
            'avg_reward_per_episode': bench[name]['avg_reward_per_episode'],
            'avg_reward_per_step': bench[name]['avg_reward_per_step'],
            'action_match_rate': bench[name]['action_match_rate'],
        })
table_policy = pd.DataFrame(policy_rows)
table_policy.to_csv(PAPER_DIR / 'table_policy_benchmark.csv', index=False)

table_training = pd.DataFrame([
    {
        'episodes': rl_summary['train_metrics']['episodes'],
        'avg_episode_reward': rl_summary['train_metrics']['avg_episode_reward'],
        'avg_episode_steps': rl_summary['train_metrics']['avg_episode_steps'],
        'final_epsilon': rl_summary['train_metrics']['final_epsilon'],
        'state_observations': rl_state_summary['observations'],
    }
])
table_training.to_csv(PAPER_DIR / 'table_training_summary.csv', index=False)

print('Saved tables to', PAPER_DIR)
table_policy


Saved tables to /Users/crishuynh/Documents/SoftwareProject/coup_detection/data/research/paper_artifacts/20260412_173937


,policy,avg_reward_per_episode,avg_reward_per_step,action_match_rate
0,dqn_policy,0.2425,0.013269,0.308482
1,random_policy,-0.9125,-0.049932,0.150479
2,behavior_cloning,0.3575,0.019562,0.324213


## 6) Create key figures

In [6]:
# Figure 1: Advisor thesis metrics (mean ± std)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(table_advisor_main['metric'], table_advisor_main['mean'], yerr=table_advisor_main['std'], capsize=4)
ax.set_title('Advisor Multi-Seed Metrics')
ax.set_ylabel('Score')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
fig.savefig(PAPER_DIR / 'fig_advisor_multiseed_metrics.png', dpi=150)
plt.close(fig)

# Figure 2: Bot-mix outcome and challenge F1
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(table_botmix))
ax.bar([i - 0.18 for i in x], table_botmix['challenge_f1_mean'], width=0.36, label='challenge_f1')
ax.bar([i + 0.18 for i in x], table_botmix['outcome_accuracy_mean'], width=0.36, label='outcome_accuracy')
ax.set_xticks(list(x))
ax.set_xticklabels(table_botmix['condition'], rotation=20)
ax.set_ylim(0.0, 1.0)
ax.set_title('Advisor Performance by Bot Mix')
ax.legend()
plt.tight_layout()
fig.savefig(PAPER_DIR / 'fig_botmix_comparison.png', dpi=150)
plt.close(fig)

# Figure 3: DQN reward curve
rl_history_df['rolling_reward_10'] = rl_history_df['reward'].rolling(window=10, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rl_history_df['episode'], rl_history_df['reward'], alpha=0.4, label='episode reward')
ax.plot(rl_history_df['episode'], rl_history_df['rolling_reward_10'], linewidth=2, label='rolling mean (10)')
ax.set_title('DQN Training Reward Curve')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward')
ax.legend()
plt.tight_layout()
fig.savefig(PAPER_DIR / 'fig_dqn_reward_curve.png', dpi=150)
plt.close(fig)

# Figure 4: Policy benchmark
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(table_policy['policy'], table_policy['avg_reward_per_episode'])
axes[0].set_title('Avg Reward Per Episode')
axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(table_policy['policy'], table_policy['action_match_rate'])
axes[1].set_title('Action Match Rate')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
fig.savefig(PAPER_DIR / 'fig_policy_benchmark.png', dpi=150)
plt.close(fig)

print('Saved figures to', PAPER_DIR)


Saved figures to /Users/crishuynh/Documents/SoftwareProject/coup_detection/data/research/paper_artifacts/20260412_173937


## 7) Build manifest for paper assets

In [7]:
manifest = {
    'sources': {
        'advisor_source': str(ADVISOR_SOURCE),
        'rl_source': str(RL_SOURCE),
    },
    'tables': sorted([p.name for p in PAPER_DIR.glob('table_*.csv')]),
    'figures': sorted([p.name for p in PAPER_DIR.glob('fig_*.png')]),
}
(PAPER_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))


{
  "sources": {
    "advisor_source": "/Users/crishuynh/Documents/SoftwareProject/coup_detection/data/research/decision",
    "rl_source": "/Users/crishuynh/Documents/SoftwareProject/coup_detection/data/research/frozen_runs/20260409_135754/dqn_rl_fit"
  },
  "tables": [
    "table_advisor_main.csv",
    "table_botmix_comparison.csv",
    "table_policy_benchmark.csv",
    "table_training_summary.csv",
    "table_weighted_ranking.csv"
  ],
  "figures": [
    "fig_advisor_multiseed_metrics.png",
    "fig_botmix_comparison.png",
    "fig_dqn_reward_curve.png",
    "fig_policy_benchmark.png"
  ]
}
